# Projet : Mise en Correspondance
**Module : Vision Artificielle · M1 IAA · USTO-MB · Année 2025/2026**

---

**Objectif :** Réaliser une mise en correspondance entre images en utilisant uniquement les contours comme primitives de détection.

---

## Pipeline complet

| Phase | Contenu | Exigence du projet |
|-------|---------|--------------------|
| **Setup** | Installation des dépendances | Prérequis |
| **Partie 1** | Chargement des images + Filtre LoG + Dilatation morphologique | Partie 1 §1 & §2 |
| **Partie 2** | SIFT masqué sur les contours + comptage des points clés | Partie 2 §1 |
| **Partie 3A** | FLANN + Ratio Test de Lowe + visualisation | Partie 3 §1 |
| **Partie 3B** | SIFT/FLANN sans masque (baseline) + comparaison | Partie 3 §2 |
| **Défense** | Questions anticipées pour la soutenance | — |

---
## ⚙️ Setup — Installation des dépendances

**Exigence du projet :** Prérequis pour toutes les parties.

Il faut la version **contrib** d'OpenCV. La version standard `opencv-python` ne contient pas SIFT dans son API publique. Il ne faut **jamais** avoir les deux installés simultanément — cela provoque des conflits d'import silencieux où `cv2.SIFT_create()` lève `AttributeError` même si l'import réussit.

> **Vérification :** `python -c "import cv2; print(cv2.__version__)"`

In [ ]:
# Désinstaller la version standard si présente, puis installer contrib
# Exécuter cette cellule une seule fois, puis redémarrer le kernel.

# !pip uninstall opencv-python -y
# !pip install opencv-contrib-python numpy matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"OpenCV version : {cv2.__version__}")
print(f"NumPy  version : {np.__version__}")

# Vérification que SIFT est disponible
try:
    _ = cv2.SIFT_create()
    print("SIFT : OK (opencv-contrib correctement installé)")
except AttributeError:
    print("ERREUR : SIFT introuvable. Installez opencv-contrib-python.")

---
## Partie 1 — Filtre LoG + Dilatation Morphologique

**Exigences du projet (Partie 1) :**
- Charger 3 images différentes du même objet ✅
- Appliquer le filtre LoG pour extraire les contours ✅
- Effectuer une dilatation morphologique sur le masque obtenu ✅
- Afficher les résultats à chaque étape ✅

---

### Justification mathématique — Laplacien de Gaussienne (LoG)

Le LoG combine deux opérations :

1. **Lissage Gaussien G(σ)** : supprime le bruit avant la dérivation. Un Laplacien appliqué directement à une image bruitée produit des résultats inutilisables.
2. **Laplacien ∇²** : opérateur de dérivée du second ordre. Ses **passages par zéro** correspondent aux vraies localisations des bords.

$$\text{LoG}(x,y) = -\frac{1}{\pi\sigma^4}\left[1 - \frac{x^2+y^2}{2\sigma^2}\right] e^{-\frac{x^2+y^2}{2\sigma^2}}$$

En OpenCV on décompose cela en `GaussianBlur` → `Laplacian`, plus contrôlable numériquement.

---

### Pourquoi la dilatation ?

SIFT évalue chaque candidat point-clé dans un voisinage patch de **16×16 pixels**. Un masque contour de 1 pixel d'épaisseur rejette des points-clés valides dont le centre détecté se trouve à 1–2 px du contour. La dilatation élargit le masque en une bande "épaisse", préservant la couverture spatiale tout en restreignant la détection aux régions structurellement pertinentes.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTIE 1 — Étape 1 : Chargement des images
# Exigence du projet : "Choisissez 3 images différentes représentant le même objet"
# ─────────────────────────────────────────────────────────────────────────────

def load_and_convert(path: str) -> tuple:
    """
    Charge une image et retourne (BGR, RGB).

    GOTCHA — BGR vs RGB :
    OpenCV lit les images en ordre de canaux BGR.
    Matplotlib attend RGB. Si on oublie cette conversion,
    les canaux rouge et bleu sont échangés. Sur des images
    quasi-monochromes (comme notre dinosaure blanc), l'erreur
    est invisible — d'où l'importance d'en faire une habitude.
    """
    bgr = cv2.imread(path)
    if bgr is None:
        raise FileNotFoundError(f"Image introuvable : {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return bgr, rgb


# ── Configurez ici les chemins vers vos 3 images ──────────────────────────────
IMAGE_PATHS = ["im1.png", "im2.png", "im3.png"]
IMAGE_NAMES = ["Vue 1", "Vue 2", "Vue 3"]
# ──────────────────────────────────────────────────────────────────────────────

bgr_images, rgb_images = [], []

for path in IMAGE_PATHS:
    bgr, rgb = load_and_convert(path)
    bgr_images.append(bgr)
    rgb_images.append(rgb)
    print(f"Chargée : {path} | Taille : {bgr.shape[1]}×{bgr.shape[0]} px")

# Affichage des images originales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Partie 1 — Images Originales", fontsize=14, fontweight="bold")
for i, (rgb, name) in enumerate(zip(rgb_images, IMAGE_NAMES)):
    axes[i].imshow(rgb)
    axes[i].set_title(name)
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("p1_originales.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTIE 1 — Étape 2 : Filtre LoG
# Exigence du projet : "Appliquez le filtre LoG pour extraire les contours"
# ─────────────────────────────────────────────────────────────────────────────

def apply_log_filter(
    bgr_img: np.ndarray,
    gauss_ksize: int = 5,
    gauss_sigma: float = 1.4,
    laplacian_ksize: int = 3,
    log_threshold: int = 15,
) -> tuple:
    """
    Applique le filtre LoG et retourne (réponse LoG uint8, masque binaire).

    Paramètres
    ----------
    gauss_ksize     : Taille du noyau Gaussien (doit être impair).
                      Plus grand → plus de flou → seuls les bords grossiers survivent.
    gauss_sigma     : σ de la Gaussienne. Règle : σ ≈ ksize/6.
                      σ=1.4 correspond au σ₀ de base de SIFT — choix intentionnel.
    laplacian_ksize : Ouverture du Laplacien discret (1, 3 ou 5).
                      ksize=3 utilise le noyau 3×3 standard :
                        [ 0  1  0]
                        [ 1 -4  1]
                        [ 0  1  0]
    log_threshold   : Seuil sur la réponse LoG pour déclarer "bord".
                      Augmenter pour obtenir moins de contours, plus forts.
    """
    # Étape 1 — Niveaux de gris
    # Le Laplacien est un opérateur scalaire ; il nécessite une entrée monocanal.
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)

    # Étape 2 — Lissage Gaussien
    # GOTCHA : ksize doit être un tuple de deux entiers IMPAIRS positifs.
    # Des valeurs paires lèvent une erreur OpenCV.
    blurred = cv2.GaussianBlur(gray, (gauss_ksize, gauss_ksize), gauss_sigma)

    # Étape 3 — Laplacien en CV_64F
    # GOTCHA CRITIQUE — toujours utiliser CV_64F (float 64 bits), jamais uint8.
    # La réponse du Laplacien est BIPOLAIRE : positive d'un côté d'un bord,
    # négative de l'autre. Avec uint8, toutes les valeurs négatives sont
    # écrêtées à 0, détruisant la moitié de l'information de bord.
    # C'est l'erreur la plus courante dans la détection de contours OpenCV.
    laplacian = cv2.Laplacian(blurred, ddepth=cv2.CV_64F, ksize=laplacian_ksize)

    # Étape 4 — Valeur absolue
    # Les deux côtés du passage par zéro deviennent positifs.
    # Les bords apparaissent comme des crêtes lumineuses.
    log_abs = np.abs(laplacian)

    # Étape 5 — Normalisation vers [0, 255] et conversion uint8
    log_norm = cv2.normalize(log_abs, None, 0, 255, cv2.NORM_MINMAX)
    log_uint8 = log_norm.astype(np.uint8)

    # Étape 6 — Seuillage binaire → masque de contours
    # Pixels > log_threshold → 255 (bord), sinon → 0.
    _, binary_mask = cv2.threshold(log_uint8, log_threshold, 255, cv2.THRESH_BINARY)

    return log_uint8, binary_mask


log_responses, binary_masks = [], []

for bgr in bgr_images:
    log_r, bin_m = apply_log_filter(bgr)
    log_responses.append(log_r)
    binary_masks.append(bin_m)

# Affichage : réponse LoG + masque binaire
fig, axes = plt.subplots(3, 2, figsize=(12, 14))
fig.suptitle("Partie 1 — Réponse LoG et Masque Binaire", fontsize=14, fontweight="bold")

for i, name in enumerate(IMAGE_NAMES):
    axes[i, 0].imshow(log_responses[i], cmap="hot")
    axes[i, 0].set_title(f"{name} — Réponse LoG |∇²Gσ|")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(binary_masks[i], cmap="gray")
    axes[i, 1].set_title(f"{name} — Masque Binaire (seuil)")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("p1_log_et_masque.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTIE 1 — Étape 3 : Dilatation Morphologique
# Exigence du projet : "effectuez une dilatation morphologique sur le masque"
# ─────────────────────────────────────────────────────────────────────────────

def apply_dilation(
    binary_mask: np.ndarray,
    dilation_ksize: int = 5,
    dilation_iter: int = 2,
) -> np.ndarray:
    """
    Applique une dilatation morphologique au masque binaire.

    Paramètres
    ----------
    dilation_ksize : Taille de l'élément structurant.
    dilation_iter  : Nombre d'itérations de dilatation.

    Note sur l'élément structurant ELLIPSE vs RECT :
    L'ellipse est plus isotrope qu'un carré. Un carré favorise
    artificiellement les directions horizontales/verticales,
    ce qui biaise l'assignation d'orientation de SIFT sur
    les bords diagonaux. L'ellipse est neutre.
    """
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (dilation_ksize, dilation_ksize)
    )
    return cv2.dilate(binary_mask, kernel, iterations=dilation_iter)


dilated_masks = [apply_dilation(mask) for mask in binary_masks]

# Affichage complet des 4 étapes côte à côte
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle(
    "Partie 1 — Pipeline Complet : Original | LoG | Masque | Masque Dilaté",
    fontsize=14, fontweight="bold"
)
col_titles = [
    "Original",
    "Réponse LoG |∇²Gσ|",
    "Masque Binaire",
    "Masque Dilaté → SIFT",
]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight="bold")

for i, name in enumerate(IMAGE_NAMES):
    axes[i, 0].imshow(rgb_images[i])
    axes[i, 1].imshow(log_responses[i], cmap="hot")
    axes[i, 2].imshow(binary_masks[i], cmap="gray")
    axes[i, 3].imshow(dilated_masks[i], cmap="gray")
    for col in range(4):
        axes[i, col].set_ylabel(name, fontsize=10)
        axes[i, col].axis("off")

plt.tight_layout()
plt.savefig("p1_pipeline_complet.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Partie 1] Terminée. Masques dilatés prêts pour la Partie 2.")

---
## Partie 2 — Détection SIFT avec Masque de Contours

**Exigences du projet (Partie 2) :**
- Appliquer SIFT sur les images **en utilisant le masque des contours** ✅
- Afficher le **nombre de points détectés** sur chaque image ✅

---

### Comment SIFT utilise le masque en interne

SIFT construit d'abord une pyramide d'échelle DoG (Différence de Gaussiennes), puis trouve les extrema locaux en espace et en échelle. La vérification du masque intervient à l'étape de sélection des candidats — tout extremum dont la position (x,y) correspond à `mask=0` est **éliminé avant** l'assignation d'orientation et le calcul du descripteur.

> **GOTCHA critique — dtype du masque :** Le masque passé à `detectAndCompute` DOIT être de dtype `np.uint8`. Un tableau booléen ou int32 lève : `(-5:Bad argument) mask is not a flag matrix`. Toujours appeler `.astype(np.uint8)` avant de passer.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTIE 2 — SIFT avec masque de contours
# Exigence : SIFT sur les images en utilisant le masque des contours
# ─────────────────────────────────────────────────────────────────────────────

def run_masked_sift(
    bgr_img: np.ndarray,
    mask: np.ndarray,
    n_features: int = 0,
) -> tuple:
    """
    Détecte les points-clés SIFT restreints aux régions non-nulles du masque.

    Paramètres
    ----------
    bgr_img    : Image originale BGR (depuis cv2.imread).
    mask       : Masque uint8 issu de la Partie 1 (dilaté).
                 255 = région candidate, 0 = ignoré.
    n_features : 0 = illimité (classé par force de réponse DoG).

    Retourne
    --------
    keypoints   : liste de cv2.KeyPoint contenant :
                  .pt (x,y), .size (échelle), .angle (orientation),
                  .response (force DoG), .octave
    descriptors : np.ndarray de forme (N, 128), dtype float32.
                  Chaque ligne = histogramme de gradients normalisé L2
                  d'un point-clé (4×4 cellules × 8 orientations = 128).

    Paramètres internes de SIFT_create (valeurs par défaut explicitées) :
      nOctaveLayers=3      : niveaux d'échelle par octave dans la pyramide DoG
      contrastThreshold=0.04 : élimine les points-clés de faible contraste
      edgeThreshold=10     : filtre Harris — rejette les réponses de type bord
      sigma=1.6            : σ de la Gaussienne de base (échelle initiale)
    """
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create(nfeatures=n_features)

    # GOTCHA : forcer uint8 sur le masque
    safe_mask = mask.astype(np.uint8)

    # detectAndCompute exécute l'intégralité du pipeline SIFT en un appel :
    # pyramide DoG → extrema → raffinement sous-pixel → orientation → descripteur
    # Seuls les extrema DANS les régions mask≠0 survivent à l'étape 3.
    keypoints, descriptors = sift.detectAndCompute(gray, safe_mask)
    return keypoints, descriptors


# ── Détection sur les 3 images ────────────────────────────────────────────────
all_keypoints, all_descriptors = [], []

print("Partie 2 — SIFT avec masque de contours :")
print("-" * 45)
for i, (bgr, mask) in enumerate(zip(bgr_images, dilated_masks)):
    kps, descs = run_masked_sift(bgr, mask)
    all_keypoints.append(kps)
    all_descriptors.append(descs)
    print(f"  {IMAGE_NAMES[i]} : {len(kps):>5} points-clés détectés")

# ── Visualisation avec points-clés RICH ──────────────────────────────────────
# DRAW_RICH_KEYPOINTS : dessine chaque point-clé comme un cercle dont :
#   rayon = échelle (size) du point-clé
#   trait = angle d'orientation dominant
# C'est informatif pour SIFT car l'échelle et l'orientation sont ses
# invariances fondamentales.

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(
    "Partie 2 — Points-Clés SIFT (masqués sur contours)",
    fontsize=14, fontweight="bold"
)

for i in range(3):
    kp_img = cv2.drawKeypoints(
        rgb_images[i],
        all_keypoints[i],
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )
    axes[i].imshow(kp_img)
    axes[i].set_title(
        f"{IMAGE_NAMES[i]}\n{len(all_keypoints[i])} points-clés",
        fontsize=12
    )
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("p2_sift_masque.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Partie 2] Terminée.")

---
## Partie 3A — Mise en Correspondance FLANN + Test du Ratio de Lowe

**Exigences du projet (Partie 3, §1) :**
- Utiliser FLANN pour rechercher les plus proches voisins entre descripteurs ✅
- Filtrer les correspondances ambiguës ✅
- Visualiser et quantifier les correspondances obtenues ✅

---

### Pourquoi FLANN plutôt que BFMatcher ?

Le **BFMatcher** (force brute) compare chaque descripteur contre tous les autres : complexité **O(N²·128)**. FLANN construit un index **KD-Tree aléatoirisé** sur l'espace à 128 dimensions, réduisant la recherche de plus proche voisin à environ **O(N log N)** — critique pour de grands ensembles d'images.

### Test du Ratio de Lowe (D. Lowe, IJCV 2004)

Pour chaque descripteur requête, récupérer ses **2 plus proches voisins** dans l'autre image. Accepter la correspondance uniquement si :

$$\frac{d(\text{meilleur})}{d(\text{2ème meilleur})} < \text{seuil} \quad (\text{typiquement } 0.75)$$

Si les deux voisins sont proches en distance, le descripteur est dans une région **peuplée** de l'espace 128-D — la correspondance est ambiguë et doit être rejetée. Lowe a validé empiriquement 0.75 comme le seuil qui maximise le rapport précision/rappel.

> **GOTCHA — structure de sortie de knnMatch :** `knnMatch(desc1, desc2, k=2)` retourne une liste de listes. Si un descripteur a moins de 2 voisins (possible sur de très petits ensembles), la liste interne a longueur 1 et dépackager `m, n = pair` lève `ValueError`. Toujours garder `if len(pair) == 2`.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTIE 3A — Correspondance FLANN + Filtre de Lowe (SIFT masqué)
# Exigence : FLANN + filtrage des correspondances ambiguës + visualisation
# ─────────────────────────────────────────────────────────────────────────────

def build_flann_matcher() -> cv2.FlannBasedMatcher:
    """
    Configure et retourne un matcher FLANN pour descripteurs SIFT (float32).

    GOTCHA — type d'index :
    SIFT → descripteurs float32 → index KD-Tree (algorithm=1).
    ORB/BRIEF → descripteurs binaires → index LSH (algorithm=6).
    Mélanger float avec LSH ou binaire avec KDTree donne des résultats
    silencieusement erronés ou des crashes.

    trees=5  : KD-Trees parallèles construits pendant l'indexation.
               Plus d'arbres = construction plus lente, requête plus précise.
    checks=50: Nœuds inspectés par requête pendant la recherche.
               Valeur standard recommandée par la documentation OpenCV.
    """
    index_params = dict(algorithm=1, trees=5)   # 1 = FLANN_INDEX_KDTREE
    search_params = dict(checks=50)
    return cv2.FlannBasedMatcher(index_params, search_params)


def match_with_lowe(
    desc1: np.ndarray,
    desc2: np.ndarray,
    matcher: cv2.FlannBasedMatcher,
    ratio_threshold: float = 0.75,
) -> list:
    """
    Matching kNN FLANN (k=2) + Test du Ratio de Lowe.

    Retourne la liste des bonnes correspondances cv2.DMatch.
    Chaque DMatch contient :
      .queryIdx  : index dans desc1
      .trainIdx  : index dans desc2
      .distance  : distance euclidienne dans l'espace 128-D (plus petit = mieux)
    """
    if desc1 is None or desc2 is None:
        return []

    # k=2 : récupérer les 2 plus proches voisins pour chaque descripteur requête.
    raw_matches = matcher.knnMatch(desc1, desc2, k=2)

    good_matches = []
    for pair in raw_matches:
        # Garde : certains descripteurs peuvent avoir moins de 2 voisins.
        if len(pair) == 2:
            m, n = pair  # m = meilleur, n = 2ème meilleur
            # Test du ratio de Lowe :
            if m.distance < ratio_threshold * n.distance:
                good_matches.append(m)

    return good_matches


def visualize_matches(
    rgb1, kp1, rgb2, kp2,
    good_matches, title, save_path,
    max_display: int = 60,
):
    """
    Dessine les lignes de correspondances entre deux images côte à côte.

    max_display : Limite le nombre de lignes affichées pour la lisibilité.
                  Afficher 500+ lignes rend l'image illisible.
                  On trie par distance pour afficher les meilleures.
    """
    display_matches = sorted(good_matches, key=lambda x: x.distance)[:max_display]

    # drawMatches attend des images BGR.
    bgr1 = cv2.cvtColor(rgb1, cv2.COLOR_RGB2BGR)
    bgr2 = cv2.cvtColor(rgb2, cv2.COLOR_RGB2BGR)

    match_img = cv2.drawMatches(
        bgr1, kp1, bgr2, kp2,
        display_matches, None,
        matchColor=(0, 255, 0),        # lignes vertes = correspondances acceptées
        singlePointColor=(255, 0, 0),  # points rouges = non appariés
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    match_rgb = cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(18, 7))
    plt.imshow(match_rgb)
    plt.title(title, fontsize=13, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Sauvegardé → {save_path}")


# ── Exécution : correspondances masquées ──────────────────────────────────────
flann = build_flann_matcher()

pairs = [(0, 1), (0, 2), (1, 2)]
pair_labels = [f"{IMAGE_NAMES[a]} ↔ {IMAGE_NAMES[b]}" for a, b in pairs]

masked_match_counts = []

print("\nPartie 3A — Correspondances FLANN (SIFT masqué) :")
print("-" * 50)

for (i, j), label in zip(pairs, pair_labels):
    good = match_with_lowe(all_descriptors[i], all_descriptors[j], flann)
    masked_match_counts.append(len(good))
    print(f"  {label} : {len(good)} bonnes correspondances (après filtre de Lowe)")
    visualize_matches(
        rgb_images[i], all_keypoints[i],
        rgb_images[j], all_keypoints[j],
        good,
        title=f"SIFT Masqué + FLANN | {label} | {len(good)} correspondances",
        save_path=f"p3a_masque_{i}{j}.png",
    )

---
## Partie 3B — Baseline : SIFT sur Image Complète (sans masque) + Comparaison

**Exigences du projet (Partie 3, §2) :**
- Répéter la procédure en appliquant SIFT directement sur les images complètes ✅
- Comparer les résultats obtenus ✅

---

### Ce que la comparaison devrait montrer

L'approche masquée produit **moins de points-clés** (les régions plates sans texture sont exclues), mais les correspondances obtenues sont de **meilleure qualité géométrique** — concentrées sur les bords structurellement significatifs. La baseline produit plus de points-clés, dont beaucoup dans des régions sans texture où les descripteurs sont non-distinctifs et les correspondances sont fortuites.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PARTIE 3B — SIFT Baseline (sans masque) + Comparaison
# Exigence : répéter SIFT/FLANN sans masque et comparer
# ─────────────────────────────────────────────────────────────────────────────

def run_baseline_sift(bgr_img: np.ndarray) -> tuple:
    """
    SIFT sur l'image complète sans masque — référence pour comparaison.
    mask=None signifie "aucune restriction", tous les pixels sont candidats.
    """
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()
    return sift.detectAndCompute(gray, None)  # None = pas de restriction


# ── Détection baseline sur les 3 images ──────────────────────────────────────
print("Partie 3B — SIFT Baseline (sans masque) :")
print("-" * 45)

baseline_kps_list, baseline_desc_list = [], []

for i, bgr in enumerate(bgr_images):
    bkp, bdesc = run_baseline_sift(bgr)
    baseline_kps_list.append(bkp)
    baseline_desc_list.append(bdesc)
    print(f"  {IMAGE_NAMES[i]} : {len(bkp):>5} points-clés détectés")

# ── Correspondances baseline ──────────────────────────────────────────────────
baseline_match_counts = []

print("\nCorrespondances FLANN baseline :")
print("-" * 45)

for (i, j), label in zip(pairs, pair_labels):
    good_b = match_with_lowe(
        baseline_desc_list[i], baseline_desc_list[j], flann
    )
    baseline_match_counts.append(len(good_b))
    print(f"  {label} : {len(good_b)} correspondances (baseline)")
    visualize_matches(
        rgb_images[i], baseline_kps_list[i],
        rgb_images[j], baseline_kps_list[j],
        good_b,
        title=f"Baseline SIFT+FLANN | {label} | {len(good_b)} correspondances",
        save_path=f"p3b_baseline_{i}{j}.png",
    )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TABLEAU DE COMPARAISON FINAL
# Exigence : comparer les résultats SIFT masqué vs SIFT complet
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 72)
print("  RÉSULTATS : SIFT Masqué (contours) vs SIFT Baseline (image complète)")
print("=" * 72)

print(f"\n{'Image':<12} {'KPs Masqués':>13} {'KPs Baseline':>14} {'Ratio':>8}")
print("-" * 52)
for name, mk, bk in zip(
    IMAGE_NAMES,
    [len(k) for k in all_keypoints],
    [len(k) for k in baseline_kps_list],
):
    ratio = mk / max(bk, 1)
    print(f"{name:<12} {mk:>13} {bk:>14} {ratio:>7.2f}x")

print(f"\n{'Paire':<20} {'Masqué':>10} {'Baseline':>10} {'Ratio':>8}")
print("-" * 52)
for label, mm, bm in zip(pair_labels, masked_match_counts, baseline_match_counts):
    ratio = mm / max(bm, 1)
    print(f"{label:<20} {mm:>10} {bm:>10} {ratio:>7.2f}x")

print("=" * 72)

# ── Visualisation comparative des counts de correspondances ──────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pair_labels))
width = 0.35

bars1 = ax.bar(x - width/2, masked_match_counts, width,
               label="SIFT Masqué (contours)", color="#1D9E75", alpha=0.85)
bars2 = ax.bar(x + width/2, baseline_match_counts, width,
               label="SIFT Baseline (image complète)", color="#378ADD", alpha=0.85)

ax.set_xlabel("Paires d'images", fontsize=12)
ax.set_ylabel("Nombre de bonnes correspondances", fontsize=12)
ax.set_title(
    "Comparaison : SIFT Masqué vs SIFT Baseline\n(après Test du Ratio de Lowe, seuil=0.75)",
    fontsize=13, fontweight="bold"
)
ax.set_xticks(x)
ax.set_xticklabels(pair_labels, fontsize=10)
ax.legend(fontsize=11)

for bar in bars1:
    ax.annotate(str(bar.get_height()),
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=10)
for bar in bars2:
    ax.annotate(str(bar.get_height()),
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("p3_comparaison.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Partie 3] Terminée. Toutes les figures sauvegardées.")

---
## Questions Anticipées pour la Soutenance

---

### Q1 — « Pourquoi LoG plutôt que Canny pour l'extraction des contours ? »

**Réponse :** Canny est un pipeline complet (magnitude du gradient + suppression non-maximale + double seuillage + hystérésis) qui produit des bords de **1 pixel d'épaisseur**. Ils sont géométriquement précis mais donnent à SIFT presque aucun support spatial — un point-clé a besoin d'un voisinage patch pour assigner son orientation et calculer son descripteur 128-D. Le LoG produit à la place une carte de réponse où les bords apparaissent comme de larges crêtes. Après seuillage et dilatation, on obtient une bande structurée plutôt qu'un trait fin. De plus, le LoG a une justification théorétique plus profonde : c'est l'opérateur naturel de second ordre dans la théorie de l'espace-échelle Gaussien (Marr & Hildreth, 1980), et SIFT lui-même est construit sur ce même cadre via l'approximation DoG du LoG.

---

### Q2 — « Pourquoi le test du ratio nécessite k=2 voisins ? Pourquoi pas juste le meilleur ? »

**Réponse :** Avec k=1 on connaît seulement la meilleure correspondance — on n'a aucun point de référence pour juger si elle est distinctive. Le test du ratio exploite la géométrie de l'espace 128-D : si un descripteur requête a deux voisins proches en distance, il se trouve dans une région peuplée et ambiguë de l'espace. Le ratio d(meilleur)/d(2ème meilleur) mesure cette ambiguïté directement. Quand il est élevé (proche de 1), les deux candidats sont également plausibles — rejeter. Quand il est faible (bien en dessous de 0.75), la meilleure correspondance est clairement dominante — accepter. Lowe a validé empiriquement 0.75 comme le seuil qui maximise le taux de correspondances correctes.

---

### Q3 — « Qu'attendez-vous de la comparaison, et pourquoi le SIFT masqué a-t-il potentiellement moins de correspondances ? »

**Réponse :** L'approche masquée devrait produire moins de points-clés globalement (les régions plates sont exclues), mais les correspondances produites sont de meilleure qualité géométrique — concentrées sur les bords significatifs. La baseline produit plus de points-clés, beaucoup dans des régions sans texture où les descripteurs sont non-distinctifs et les correspondances sont fortuites. Le test du ratio en supprime une partie, mais l'approche masquée empêche structurellement les candidats de faible qualité d'être générés. Si le nombre de correspondances masquées est significativement inférieur à la baseline, cela reflète la rareté des régions de contours informatifs dans l'image.

---

### Q4 — « Pourquoi avez-vous utilisé CV_64F dans l'appel Laplacian ? »

**Réponse :** Le Laplacien est une dérivée du second ordre et sa sortie est bipolaire — elle devient négative d'un côté d'un bord et positive de l'autre. Avec uint8 (non signé 8 bits), toutes les valeurs négatives sont écrêtées à 0, détruisant exactement la moitié de l'information de bord (un côté complet de chaque passage par zéro). Avec CV_64F (float 64 bits) on préserve la plage signée complète. On prend ensuite la valeur absolue, ce qui mappe les deux côtés du passage par zéro vers des valeurs positives et donne une réponse symétrique et lumineuse à chaque bord.